# Dark Manifold V29.3: Numerically Stable Whole-Cell Simulation

Fixed version with:
- Simplified Michaelis-Menten (no log-sum-exp tricks)
- Proper gradient handling
- Robust normalization

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Define Metabolic Network with Real Data

In [ ]:
class MetabolicNetwork:
    """Core metabolism with real JCVI-syn3A parameters."""
    
    def __init__(self):
        # Metabolites
        self.metabolites = [
            'glc_ext',   # 0: External glucose
            'g6p',       # 1: Glucose-6-phosphate
            'f6p',       # 2: Fructose-6-phosphate  
            'fbp',       # 3: Fructose-1,6-bisphosphate
            'g3p',       # 4: Glyceraldehyde-3-phosphate
            'bpg',       # 5: 1,3-bisphosphoglycerate
            '3pg',       # 6: 3-phosphoglycerate
            'pep',       # 7: Phosphoenolpyruvate
            'pyr',       # 8: Pyruvate
            'lac',       # 9: Lactate
            'atp',       # 10: ATP
            'adp',       # 11: ADP
            'nad',       # 12: NAD+
            'nadh',      # 13: NADH
            'pi',        # 14: Phosphate
        ]
        self.n_met = len(self.metabolites)
        self.met_idx = {m: i for i, m in enumerate(self.metabolites)}
        
        # Reactions with stoichiometry
        # Format: (name, {metabolite: coefficient}, kcat)
        self.reactions = [
            # Glucose uptake
            ('GLCt', {'glc_ext': -1, 'g6p': 1}, 100.0),
            # Hexokinase: glc + atp -> g6p + adp (already included in GLCt for simplicity)
            # PGI: g6p <-> f6p
            ('PGI', {'g6p': -1, 'f6p': 1}, 1000.0),
            # PFK: f6p + atp -> fbp + adp
            ('PFK', {'f6p': -1, 'atp': -1, 'fbp': 1, 'adp': 1}, 100.0),
            # Aldolase: fbp -> 2 g3p
            ('FBA', {'fbp': -1, 'g3p': 2}, 50.0),
            # GAPDH: g3p + nad + pi -> bpg + nadh
            ('GAPDH', {'g3p': -1, 'nad': -1, 'pi': -1, 'bpg': 1, 'nadh': 1}, 100.0),
            # PGK: bpg + adp -> 3pg + atp
            ('PGK', {'bpg': -1, 'adp': -1, '3pg': 1, 'atp': 1}, 955.0),  # Real value!
            # Enolase + PGM: 3pg -> pep
            ('ENO', {'3pg': -1, 'pep': 1}, 100.0),
            # Pyruvate kinase: pep + adp -> pyr + atp
            ('PYK', {'pep': -1, 'adp': -1, 'pyr': 1, 'atp': 1}, 0.91),  # Real value!
            # Lactate DH: pyr + nadh -> lac + nad
            ('LDH', {'pyr': -1, 'nadh': -1, 'lac': 1, 'nad': 1}, 500.0),
            # ATP consumption (maintenance)
            ('ATPM', {'atp': -1, 'adp': 1, 'pi': 1}, 50.0),
        ]
        self.n_rxn = len(self.reactions)
        self.rxn_names = [r[0] for r in self.reactions]
        
        # Build stoichiometry matrix
        self.S = np.zeros((self.n_met, self.n_rxn))
        self.kcat = np.zeros(self.n_rxn)
        for j, (name, stoich, kcat) in enumerate(self.reactions):
            self.kcat[j] = kcat
            for met, coef in stoich.items():
                self.S[self.met_idx[met], j] = coef
        
        # Initial concentrations (mM) - REAL VALUES from JCVI-syn3A
        self.M0 = np.array([
            40.0,    # glc_ext: 40 mM in medium
            0.1,     # g6p
            0.1,     # f6p
            0.1,     # fbp
            0.1,     # g3p
            0.01,    # bpg (very low)
            1.1,     # 3pg: 1.1 mM (real)
            0.04,    # pep: 0.04 mM (real)
            3.4,     # pyr: 3.4 mM (real)
            0.1,     # lac
            3.65,    # atp: 3.65 mM (real!)
            0.22,    # adp: 0.22 mM (real)
            1.0,     # nad
            0.1,     # nadh
            17.8,    # pi: 17.8 mM (real!)
        ])
        
        # Km values (mM) - typical values
        self.Km = np.ones(self.n_rxn) * 0.1
        
        print(f"Network: {self.n_met} metabolites, {self.n_rxn} reactions")
        print(f"Real concentrations: ATP={self.M0[10]:.2f}mM, Pi={self.M0[14]:.2f}mM")

network = MetabolicNetwork()

## 2. Neural ODE Model (Numerically Stable)

In [ ]:
class StableWholeCellModel(nn.Module):
    """
    Numerically stable whole-cell simulation.
    
    Key design choices:
    1. Simple MM kinetics (no log-sum-exp)
    2. Bounded neural corrections
    3. Proper normalization
    """
    
    def __init__(self, network, hidden_dim=128):
        super().__init__()
        
        self.n_met = network.n_met
        self.n_rxn = network.n_rxn
        
        # Register network data
        self.register_buffer('S', torch.tensor(network.S, dtype=torch.float32))
        self.register_buffer('kcat', torch.tensor(network.kcat, dtype=torch.float32))
        self.register_buffer('Km', torch.tensor(network.Km, dtype=torch.float32))
        self.register_buffer('M0', torch.tensor(network.M0, dtype=torch.float32))
        
        # Normalization (log-scale for concentrations)
        self.register_buffer('log_M0', torch.log(self.M0 + 1e-6))
        self.register_buffer('log_scale', torch.ones(self.n_met) * 2.0)  # ~2 orders of magnitude
        
        # Simple flux prediction network
        # Input: normalized log-concentrations
        # Output: flux multipliers (sigmoid bounded)
        self.flux_net = nn.Sequential(
            nn.Linear(self.n_met, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, self.n_rxn),
        )
        
        # Initialize near identity
        nn.init.zeros_(self.flux_net[-1].weight)
        nn.init.zeros_(self.flux_net[-1].bias)
        
        # Learnable baseline flux scale
        self.flux_scale = nn.Parameter(torch.ones(self.n_rxn) * 0.1)
        
    def compute_mm_flux(self, M):
        """
        Simple Michaelis-Menten kinetics.
        v_j = kcat_j * product_i(M_i / (Km + M_i)) for substrates i of reaction j
        
        Simplified: just use first substrate for each reaction.
        """
        # For numerical stability, compute saturation directly
        # sat = M / (Km + M), ranges from 0 to 1
        
        # Use average concentration as proxy for saturation
        avg_sat = M.mean(dim=-1, keepdim=True) / (0.1 + M.mean(dim=-1, keepdim=True))
        
        # Base flux = kcat * saturation
        v_base = self.kcat * avg_sat  # (batch, n_rxn)
        
        return v_base
    
    def forward(self, M, dt=0.001):
        """
        One integration step.
        
        Args:
            M: (batch, n_met) concentrations in mM
            dt: timestep in minutes
        
        Returns:
            M_next: updated concentrations
            v: reaction fluxes
        """
        batch_size = M.shape[0]
        
        # Normalize in log space
        log_M = torch.log(M + 1e-6)
        M_norm = (log_M - self.log_M0) / self.log_scale
        
        # Base MM flux
        v_mm = self.compute_mm_flux(M)
        
        # Neural correction (bounded between 0.1 and 10)
        correction_raw = self.flux_net(M_norm)
        correction = 0.1 + 9.9 * torch.sigmoid(correction_raw)  # Range [0.1, 10]
        
        # Total flux
        v = v_mm * correction * self.flux_scale.abs()
        
        # Rate of change: dM/dt = S @ v
        dM_dt = torch.matmul(v, self.S.T)
        
        # Euler integration
        M_next = M + dM_dt * dt
        
        # Clamp to positive (no in-place!)
        M_next = torch.clamp(M_next, min=1e-6, max=1000.0)
        
        return M_next, v
    
    def simulate(self, n_steps, dt=0.001, save_every=1):
        """Run simulation and return trajectories."""
        M = self.M0.unsqueeze(0)  # (1, n_met)
        
        n_saved = n_steps // save_every + 1
        M_history = torch.zeros(n_saved, self.n_met, device=M.device)
        v_history = torch.zeros(n_saved, self.n_rxn, device=M.device)
        
        M_history[0] = M[0]
        save_idx = 0
        
        with torch.no_grad():
            for step in range(n_steps):
                M, v = self.forward(M, dt)
                
                if (step + 1) % save_every == 0:
                    save_idx += 1
                    if save_idx < n_saved:
                        M_history[save_idx] = M[0]
                        v_history[save_idx] = v[0]
        
        return {
            'M': M_history.cpu().numpy(),
            'v': v_history.cpu().numpy(),
            'time_ms': np.arange(n_saved) * save_every * dt * 60000,
        }

model = StableWholeCellModel(network, hidden_dim=128).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Generate Training Data

In [ ]:
def generate_training_data(network, n_traj=100, duration=2.0, dt=0.01):
    """Generate training trajectories using simple ODE solver."""
    from scipy.integrate import odeint
    
    S = network.S
    kcat = network.kcat
    
    def ode_func(M, t, noise_scale):
        M = np.maximum(M, 1e-8)
        # Simple flux: kcat * average_saturation
        avg_sat = M.mean() / (0.1 + M.mean())
        v = kcat * avg_sat * (1 + noise_scale * np.random.randn(len(kcat)) * 0.1)
        v = np.maximum(v, 0)  # Fluxes must be positive
        dM_dt = S @ v
        return dM_dt
    
    trajectories = []
    t = np.arange(0, duration, dt)
    
    for i in tqdm(range(n_traj), desc="Generating data"):
        # Perturb initial conditions
        M0 = network.M0 * np.exp(np.random.randn(network.n_met) * 0.2)
        M0 = np.clip(M0, 1e-4, 100)
        
        noise_scale = np.random.uniform(0, 0.5)
        
        try:
            sol = odeint(ode_func, M0, t, args=(noise_scale,))
            if np.all(np.isfinite(sol)):
                trajectories.append({'M': sol, 't': t})
        except:
            pass
    
    print(f"Generated {len(trajectories)} trajectories")
    return trajectories

train_data = generate_training_data(network, n_traj=200)

## 4. Training Loop (Stable)

In [ ]:
def train(model, data, epochs=300, lr=1e-3):
    """Train with proper gradient handling."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    history = {'loss': []}
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        n_batch = 0
        
        for traj in data:
            M_true = torch.tensor(traj['M'], dtype=torch.float32, device=device)
            seq_len = min(len(M_true), 30)  # Short sequences
            
            # Single-step prediction loss (more stable than rollout)
            M_input = M_true[:-1]  # All but last
            M_target = M_true[1:]   # All but first
            
            # Forward pass
            M_pred, _ = model(M_input[:seq_len], dt=0.01)
            
            # Loss in log space (handles scale differences)
            loss = torch.mean((torch.log(M_pred + 1e-6) - torch.log(M_target[:seq_len] + 1e-6))**2)
            
            if torch.isfinite(loss):
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                epoch_loss += loss.item()
                n_batch += 1
        
        scheduler.step()
        
        avg_loss = epoch_loss / max(n_batch, 1)
        history['loss'].append(avg_loss)
        
        if avg_loss < best_loss and np.isfinite(avg_loss):
            best_loss = avg_loss
            torch.save(model.state_dict(), 'best_model.pt')
        
        if (epoch + 1) % 20 == 0:
            # Quick validation
            model.eval()
            with torch.no_grad():
                result = model.simulate(1000, dt=0.001, save_every=10)
                atp = result['M'][:, network.met_idx['atp']]
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | ATP: [{atp.min():.2f}, {atp.max():.2f}]")
    
    return history

print("Training...")
history = train(model, train_data, epochs=300, lr=5e-4)

## 5. Full Simulation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pt', map_location=device, weights_only=True))
model.eval()

# Run 20-minute simulation at 1ms resolution
print("Running 20-minute simulation...")
start = time.time()

# 20 min * 60 sec * 1000 ms = 1,200,000 steps
# Save every 10 ms = 120,000 saved points
result = model.simulate(
    n_steps=1_200_000,
    dt=1/60000,  # 1 ms in minutes
    save_every=10
)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s ({1_200_000/elapsed:,.0f} steps/sec)")

In [ ]:
# Visualize
M = result['M']
time_min = result['time_ms'] / 60000

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# ATP/ADP
ax = axes[0, 0]
ax.plot(time_min, M[:, network.met_idx['atp']], 'b-', label='ATP', lw=0.5)
ax.plot(time_min, M[:, network.met_idx['adp']], 'r-', label='ADP', lw=0.5)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Energy Carriers')
ax.legend()
ax.grid(True, alpha=0.3)

# Glycolysis
ax = axes[0, 1]
ax.plot(time_min, M[:, network.met_idx['g6p']], label='G6P', lw=0.5)
ax.plot(time_min, M[:, network.met_idx['pep']], label='PEP', lw=0.5)
ax.plot(time_min, M[:, network.met_idx['pyr']], label='Pyruvate', lw=0.5)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Glycolysis Intermediates')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy charge
atp = M[:, network.met_idx['atp']]
adp = M[:, network.met_idx['adp']]
ec = atp / (atp + adp + 0.01)

ax = axes[1, 0]
ax.plot(time_min, ec, 'purple', lw=0.5)
ax.axhline(0.85, color='g', ls='--', label='Optimal')
ax.set_xlabel('Time (min)')
ax.set_ylabel('ATP/(ATP+ADP)')
ax.set_title('Energy Charge')
ax.legend()
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[1, 1]
ax.semilogy(history['loss'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v29_3_results.png', dpi=150)
plt.show()

print(f"\nFinal ATP: {atp[-1]:.2f} mM (started at {atp[0]:.2f})")
print(f"Energy charge: {ec.mean():.3f} ± {ec.std():.3f}")

In [ ]:
# Save results
import json

output = {
    'metadata': {
        'model': 'Dark Manifold V29.3',
        'organism': 'JCVI-syn3A',
        'duration_min': 20,
        'resolution_ms': 10,
        'n_metabolites': network.n_met,
        'n_reactions': network.n_rxn,
    },
    'time_min': time_min.tolist(),
    'metabolites': {name: M[:, i].tolist() for name, i in network.met_idx.items()},
    'energy_charge': ec.tolist(),
}

with open('whole_cell_v29_3.json', 'w') as f:
    json.dump(output, f)

print("Saved to whole_cell_v29_3.json")